# 232 — High-Gamma (HG) clustering

Collapses each ERSP's freq axis to a single high-gamma band (70–150 Hz default) time series,
then clusters on the resulting (n_samples × n_time) matrix.
Operates on the **canonical dataset** — same samples as 210/230/231.
Outputs land in `outputs/clustering/{kmeans,hierarchical}/hg/runs/<timestamp>/`.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions import lf_blob_clustering_config as cfg
from functions.lf_hg import (
    build_hg_feature_matrix,
    save_sample_hg_png,
    render_hg_sparkline,
)
from functions import lf_cluster_run as R

SCRIPT_NAME = '232_hg_clustering.ipynb'


## Config

In [ ]:
# ── data input ───────────────────────────────
INPUT_DIR = Path(r'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\01_FBM_Analysis\\outputs\\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

# ── HG band ──────────────────────────────────
HG_BAND = (70.0, 150.0)
FMAX    = 500.0          # ERSP freq axis ceiling (Hz)

# ── clustering ───────────────────────────────
KMEANS_K_RANGE = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
HC_METHOD      = 'ward'
HC_METRIC      = 'euclidean'
RANDOM_STATE   = cfg.RANDOM_STATE

print('HG_BAND:', HG_BAND, 'Hz   FMAX:', FMAX, 'Hz')
print('KMEANS_K_RANGE:', KMEANS_K_RANGE)


## Load canonical dataset

Shared across 210/230/231/232 so cross-feature comparisons are valid.


In [ ]:
# Loads ERSPs from INPUT_DIR, drops non-neural channels, applies the
# high-activity gate. Single source of truth so all four clustering
# notebooks operate on the same sample set. Cached in
# 02_FBM_Clustering/outputs/_dataset/canonical/ for fast reload.
from functions.lf_dataset import prepare_dataset, DEFAULT_CACHE_DIR

df_meta, ersp_list, X_3d = prepare_dataset(INPUT_DIR, cache_dir=DEFAULT_CACHE_DIR)
print(f'\nCanonical dataset: {len(df_meta)} samples · X_3d.shape={X_3d.shape}')
df_meta.head()


## Build HG feature matrix

In [ ]:
X_hg = build_hg_feature_matrix(ersp_list, hg_band=HG_BAND, fmax=FMAX)
print('X_hg:', X_hg.shape, '  dtype:', X_hg.dtype)


## Per-sample HG sparkline PNGs

Writes one PNG per sample under `04_ersp_LM_RAWONLY/<pid>/LM/ERSP_hg/<cond>/<stem>_HG.png`
for the MOBA samples-pane HG view.


In [ ]:
GEN_HG_PNGS = True
FORCE_REGEN = False

def _hg_view_path(file_path):
    p = str(file_path).replace('\\\\', '/').replace('\\', '/')
    if '/ERSP_matrix/' not in p:
        return None
    return Path(p.replace('/ERSP_matrix/', '/ERSP_hg/').replace('.npy', '_HG.png'))

if not GEN_HG_PNGS:
    print('[skip] GEN_HG_PNGS = False')
else:
    n_total = len(ersp_list)
    n_wrote = n_skip = n_bad = 0
    print(f'Writing per-sample HG sparkline PNGs for {n_total} canonical samples...')
    for i in range(n_total):
        fp = df_meta.iloc[i].get('file_path')
        if not fp:
            n_bad += 1; continue
        out_path = _hg_view_path(fp)
        if out_path is None:
            n_bad += 1; continue
        try:
            if FORCE_REGEN or not out_path.exists():
                save_sample_hg_png(ersp_list[i], out_path, hg_band=HG_BAND, fmax=FMAX)
                n_wrote += 1
            else:
                n_skip += 1
        except Exception as e:
            print(f'  [warn] sample {i}: {e}')
            n_bad += 1
        if (i + 1) % 500 == 0:
            print(f'  ...{i+1}/{n_total}')
    print(f'HG PNGs : wrote {n_wrote}, skipped existing {n_skip}')
    if n_bad:
        print(f'  ({n_bad} samples without parseable file_path — skipped)')


# Clustering

In [ ]:
manifest_km = R.fit_and_save(
    X_hg,
    df_keep=df_meta,
    method='kmeans',
    feature_set='hg',
    params={'k_range': KMEANS_K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means',
    feature_set_label='High-Gamma Time Series',
    notebook=SCRIPT_NAME,
)
BEST_K = manifest_km['summary']['best_k']
print(f'Best K (KMeans/hg, by silhouette): {BEST_K}')


In [ ]:
manifest_hc = R.fit_and_save(
    X_hg,
    df_keep=df_meta,
    method='hierarchical',
    feature_set='hg',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': KMEANS_K_RANGE},
    method_label='Hierarchical',
    feature_set_label='High-Gamma Time Series',
    notebook=SCRIPT_NAME,
)
print(f'Best K (HC/hg, by silhouette): {manifest_hc["summary"]["best_k"]}')


## Per-cluster centroid sparklines (for the MOBA cluster chips)

Mean HG time series across cluster members, rendered as a sparkline.


In [ ]:
import json
CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
INDEX_PATH = CLUSTERING_DIR / 'index.json'

def _save_per_cluster_centroid_pngs_hg(manifest, X_local):
    if manifest['feature_set'] != 'hg':
        return 0
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] / 'runs' / manifest['run_id']
    df = pd.read_csv(run_dir / 'labels.csv')
    cluster_col = f"cluster_{manifest['method']}_{manifest['feature_set']}"
    if cluster_col not in df.columns:
        cands = [c for c in df.columns if c.startswith('cluster_')]
        if not cands: return 0
        cluster_col = cands[0]
    labels = df[cluster_col].to_numpy()
    if len(labels) != X_local.shape[0]:
        print(f"  [skip] {manifest['run_id']}: labels ({len(labels)}) vs X_hg ({X_local.shape[0]}) mismatch")
        return 0
    out_dir = run_dir / 'cluster_centroids'
    out_dir.mkdir(parents=True, exist_ok=True)
    uniq = sorted(int(c) for c in np.unique(labels))
    for c in uniq:
        idx = np.where(labels == c)[0]
        mean_hg = X_local[idx].mean(axis=0)
        fig, ax = plt.subplots(figsize=(2.4, 1.7))
        render_hg_sparkline(ax, mean_hg, ylim=(-6, 6))
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        fig.savefig(out_dir / f'cluster_{int(c):02d}.png', dpi=90, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    return len(uniq)

if INDEX_PATH.exists():
    with open(INDEX_PATH) as f:
        idx_data = json.load(f)
    runs = [r for r in idx_data.get('runs', []) if r['feature_set'] == 'hg']
    print(f'Backfilling for {len(runs)} hg runs...')
    for run in runs:
        mp = CLUSTERING_DIR / run['path'] / 'manifest.json'
        if not mp.exists(): continue
        manifest = json.loads(mp.read_text())
        n = _save_per_cluster_centroid_pngs_hg(manifest, X_hg)
        if n: print(f"  [{manifest['method']}/hg] {manifest['run_id']} -> {n} PNGs")
    print('Done.')
